# Treino de um Modelo (pipeline real)

Treina pelo `TrainingService` — o MESMO caminho usado pela API e pelo
benchmark — a partir de um `.npz` (`X_train`/`y_train`). Salva o modelo
como artefato de inferência (sem estado do otimizador) com o
`input_contract` que garante paridade treino↔inferência.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

_REPO_URL = "https://github.com/thierrybraga/XFakeSong.git"


def _project_root() -> Path:
    p = Path.cwd()
    while not (p / "app").exists() and p.parent != p:
        p = p.parent
    return p


# Colab/Kaggle limpos: clona o repositório se o projeto não estiver no disco.
if not (_project_root() / "app").exists():
    if not Path("XFakeSong").exists():
        print("Clonando XFakeSong...")
        subprocess.run(["git", "clone", "--depth", "1", _REPO_URL], check=True)
    os.chdir("XFakeSong")

_DEPS = {"librosa": "librosa>=0.10", "soundfile": "soundfile>=0.12",
         "pywt": "PyWavelets>=1.4"}
_missing = [s for m, s in _DEPS.items() if importlib.util.find_spec(m) is None]
if _missing:
    print("Instalando dependências de áudio:", *_missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing],
                   check=True)

ROOT = _project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT, "| Colab:", "google.colab" in sys.modules)

## Reprodutibilidade (semente numpy + TensorFlow)

In [ ]:
import numpy as np

try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(0)
except Exception:  # pragma: no cover
    np.random.seed(0)

## 1. Dataset sintético `.npz` (espectrograma pequeno)

In [ ]:
import tempfile
import numpy as np

rng = np.random.default_rng(0)
n, T, F = 300, 32, 16
X = rng.standard_normal((n, T, F)).astype("float32")
y = rng.integers(0, 2, n).astype("int64")
X[y == 1] += 0.6  # separa as classes → treinável em poucas épocas

workdir = Path(tempfile.mkdtemp(prefix="nb_train_"))
npz = workdir / "dataset.npz"
np.savez(npz, X_train=X, y_train=y)
print("dataset:", npz)

## 2. Treinar via TrainingService

In [ ]:
from app.core.interfaces.base import ProcessingStatus
from app.domain.services.training_service import TrainingService

models_dir = workdir / "models"
svc = TrainingService(models_dir=str(models_dir))
res = svc.train_model(
    architecture="MultiscaleCNN",
    dataset_path=str(npz),
    config={"epochs": 2, "batch_size": 32, "model_name": "nb_demo"},
)
assert res.status == ProcessingStatus.SUCCESS, res.errors
md_ = res.data
print("acurácia:", round(md_.accuracy, 4))
print("artefatos:", sorted(p.name for p in models_dir.glob("nb_demo*")))

## Notas

- O `.keras` é salvo **sem o estado do otimizador** (~3× menor, load mais
  rápido, saída idêntica) — ver `trainer.save_inference_keras`.
- O `nb_demo_config.json` carrega o `input_contract` (temperatura/EER/OOD
  calibrados) lido na inferência. Ver `notebooks/pipeline/03_inference.ipynb`.

## 3. Treinar com um dataset REAL (download) — opcional

Baixa um dataset real do HuggingFace, monta o `.npz` e treina pelo MESMO
`TrainingService`. Desligado por padrão (download + treino são pesados).

> ⚠️ **Requer `datasets<4.0`** — a 4.x exige `torchcodec` e quebra o
> download de áudio (a célula instala a versão certa). Algumas fontes
> pedem um **token do HuggingFace** (erro 401) — exporte `HF_TOKEN` antes
> ou veja `docs/12_DATASETS.md`. **FLEURS** (real, PT-BR) é público e leve;
> fontes *fake* como WaveFake são grandes (dezenas de GB).

In [ ]:
import subprocess

DOWNLOAD_REAL = False          # → True para baixar + treinar com dados reais
N_PER_CLASS = 150              # amostras por classe (pequeno p/ demonstração)
REAL_NPZ = ROOT / "app" / "datasets" / "nb_real.npz"

if DOWNLOAD_REAL:
    # 1. datasets compatível (a 4.x quebra o download de áudio).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "datasets>=2.14,<4.0"], check=True)
    # 2. Pipeline canônico: download → preparo → split → export .npz → treino.
    cmd = [
        sys.executable, str(ROOT / "scripts" / "run_tcc_pipeline.py"),
        "--download", "--skip-real-cv",
        "--archs", "SVM",
        "--target-per-class", str(N_PER_CLASS),
        "--epochs", "5",
        "--npz", str(REAL_NPZ),
        "--out", str(ROOT / "results" / "nb_real"),
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)
    print("NPZ real:", REAL_NPZ, "| existe:", REAL_NPZ.exists())
else:
    print("DOWNLOAD_REAL=False — usando o dataset sintético da Seção 1.")
    print("Defina DOWNLOAD_REAL=True para baixar dados reais (ver docs/12).")

Com o `.npz` real em mãos, treine **qualquer arquitetura** repetindo a
Seção 2 com `dataset_path=str(REAL_NPZ)`. Para baixar uma fonte específica
direto (sem o pipeline completo):

```bash
pip install 'datasets>=2.14,<4.0'                                  # obrigatório
python scripts/download_datasets.py --fleurs --max-samples 300     # real (público)
python scripts/download_datasets.py --brspeech --max-samples 600   # real+fake (pode pedir token)
```